# NuevaMente — Evaluación Retrieval V2

## Objetivo

Evaluar **Retrieval V2 de Agentes** usando exactamente las mismas 50 preguntas y el mismo `ground_truth_v2.csv` utilizados como referencia, manteniendo comparabilidad con Retrieval V1.

Este notebook valida:

- uso de la query original del benchmark;
- ausencia del fallback genérico;
- trazabilidad de filtros (`none`, `document_id`, `categoria`, `document_id+categoria`);
- Recall@3, Recall@5, Precision@3 y Precision@5;
- impacto sobre los 11 casos críticos de V1;
- posibles regresiones por categoría;
- efecto observable de filtros, BM25, RRF y Cross-Encoder.

> **Importante:** este notebook no modifica el Ground Truth ni sobrescribe resultados V1.


## 1. Configuración y rutas

El notebook asume que se ejecuta desde `Data_IA/notebooks/`.

Entradas:

- `data/evaluation/ground_truth_v2.csv`
- `data/evaluation/input/retrieval_results_agentes_v2.json`
- `data/evaluation/results/retrieval_results_gt_v2.csv`
- `data/evaluation/results/retrieval_category_summary_gt_v2.csv`
- `data/evaluation/results/retrieval_error_analysis_gt_v2.csv`
- `data/evaluation/results/retrieval_v2/run_manifest.json`

Salidas:

`data/evaluation/results/retrieval_v2/`


In [1]:
from pathlib import Path
import json
import ast
import math
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
EVALUATION_DIR = DATA_DIR / "evaluation"
INPUT_DIR = EVALUATION_DIR / "input"
BASELINE_RESULTS_DIR = EVALUATION_DIR / "results"
OUTPUT_DIR = BASELINE_RESULTS_DIR / "retrieval_v2"

GROUND_TRUTH_PATH = EVALUATION_DIR / "ground_truth_v2.csv"
RETRIEVAL_V2_INPUT_PATH = INPUT_DIR / "retrieval_results_agentes_v2.json"

BASELINE_RESULTS_PATH = BASELINE_RESULTS_DIR / "retrieval_results_gt_v2.csv"
BASELINE_CATEGORY_PATH = BASELINE_RESULTS_DIR / "retrieval_category_summary_gt_v2.csv"
BASELINE_ERROR_ANALYSIS_PATH = BASELINE_RESULTS_DIR / "retrieval_error_analysis_gt_v2.csv"

MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA
OUTPUT_DIR: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2


In [2]:
required_paths = [
    GROUND_TRUTH_PATH,
    BASELINE_RESULTS_PATH,
    BASELINE_CATEGORY_PATH,
    BASELINE_ERROR_ANALYSIS_PATH,
    MANIFEST_PATH,
]

for path in required_paths:
    print("✅" if path.exists() else "❌", path)

if not RETRIEVAL_V2_INPUT_PATH.exists() or RETRIEVAL_V2_INPUT_PATH.stat().st_size == 0:
    print("\n⚠️ retrieval_results_agentes_v2.json todavía no está disponible o está vacío.")
    print("El notebook queda preparado para ejecutar la evaluación cuando llegue la corrida V2.")
else:
    print("✅", RETRIEVAL_V2_INPUT_PATH)


✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\ground_truth_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_results_gt_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_category_summary_gt_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_error_analysis_gt_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\run_manifest.json
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\input\retrieval_results_agentes_v2.json


## 2. Cargar y validar Ground Truth v2


In [3]:
ground_truth_df = pd.read_csv(GROUND_TRUTH_PATH, encoding="utf-8-sig")

assert len(ground_truth_df) == 50, "El Ground Truth debe contener exactamente 50 casos."
assert ground_truth_df["case_id"].nunique() == 50, "Los case_id deben ser únicos."
assert ground_truth_df["pregunta"].notna().all(), "Hay preguntas vacías."
assert ground_truth_df["relevant_chunk_ids"].notna().all(), "Hay casos sin relevant_chunk_ids."

print("✅ Ground Truth v2 validado")
print("Casos:", len(ground_truth_df))
print("Categorías:", sorted(ground_truth_df["categoria"].dropna().unique().tolist()))

ground_truth_df[
    ["case_id", "document_id", "categoria", "pregunta", "relevant_chunk_ids"]
].head()


✅ Ground Truth v2 validado
Casos: 50
Categorías: ['Backend', 'Cloud / DevOps', 'Data Science', 'Frontend', 'IA']


,case_id,document_id,categoria,pregunta,relevant_chunk_ids
0,AI-ES-001-Q01,AI-ES-001,IA,¿Qué pretende introducir este módulo sobre int...,AI-ES-001_CH_001
1,AI-ES-001-Q02,AI-ES-001,IA,¿Es obligatorio conocer aprendizaje automático...,AI-ES-001_CH_001
2,AI-ES-001-Q03,AI-ES-001,IA,Menciona dos cargas de trabajo de IA tratadas ...,AI-ES-001_CH_002
3,AI-ES-001-Q04,AI-ES-001,IA,¿Qué tema del módulo aborda el uso ético o seg...,AI-ES-001_CH_002
4,AI-ES-001-Q05,AI-ES-001,IA,¿Qué conocimiento previo puede ser útil antes ...,AI-ES-001_CH_001


In [4]:
with MANIFEST_PATH.open("r", encoding="utf-8-sig") as f:
    manifest = json.load(f)

assert manifest["total_cases_expected"] == 50
assert manifest["ground_truth_frozen"] is True
assert manifest["queries_frozen"] is True
assert manifest["query_policy"]["use_original_query"] is True
assert manifest["query_policy"]["fallback_allowed"] is False

print("✅ run_manifest.json validado")
manifest


✅ run_manifest.json validado


{'benchmark_name': 'NuevaMente Retrieval Benchmark',
 'benchmark_run': 'retrieval_v2_gt_v2',
 'benchmark_version': '2.0',
 'ground_truth_file': 'Data_IA/data/evaluation/ground_truth_v2.csv',
 'corpus_file': 'Data_IA/data/evaluation/chunks_v1.csv',
 'retrieval_input_file': 'Data_IA/data/evaluation/input/retrieval_results_agentes_v2.json',
 'baseline_results_file': 'Data_IA/data/evaluation/results/retrieval_results_gt_v2.csv',
 'total_cases_expected': 50,
 'ground_truth_frozen': True,
 'queries_frozen': True,
 'query_policy': {'source': 'ground_truth_v2.pregunta',
  'use_original_query': True,
  'execution_method': 'AgentV1.answer_for_evaluation',
  'fallback_allowed': False,
  'fallback_pattern': 'Conceptos principales sobre'},
 'retrieval_v2': {'metadata_filters_enabled': True,
  'filter_trace_required': True,
  'supported_filter_modes': ['none',
   'document_id',
   'categoria',
   'document_id+categoria'],
  'vector_search': True,
  'lexical_search_bm25': True,
  'rrf_enabled': True,

## 3. Funciones auxiliares


In [5]:
FALLBACK_PATTERN = manifest["query_policy"].get(
    "fallback_pattern",
    "Conceptos principales sobre"
)

def parse_chunk_ids(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []

    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    text = str(value).strip()
    if not text:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except (ValueError, SyntaxError):
            pass

    if ";" in text:
        return [x.strip() for x in text.split(";") if x.strip()]

    return [text]

def recall_at_k(relevant, retrieved, k):
    relevant_set = set(relevant)
    if not relevant_set:
        return 0.0
    return len(relevant_set.intersection(retrieved[:k])) / len(relevant_set)

def precision_at_k(relevant, retrieved, k):
    if k <= 0:
        return 0.0
    relevant_set = set(relevant)
    return len(relevant_set.intersection(retrieved[:k])) / k

def infer_filter_trace(case_payload):
    trace = case_payload.get("filter_trace") or case_payload.get("filters") or {}

    document_id = trace.get("document_id") if isinstance(trace, dict) else None
    categoria = trace.get("categoria") if isinstance(trace, dict) else None

    explicit_mode = case_payload.get("filter_mode")

    if explicit_mode:
        mode = explicit_mode
    elif document_id and categoria:
        mode = "document_id+categoria"
    elif document_id:
        mode = "document_id"
    elif categoria:
        mode = "categoria"
    else:
        mode = "none"

    return mode, document_id, categoria

def is_fallback_query(query):
    if query is None:
        return False
    return FALLBACK_PATTERN.lower() in str(query).lower()


## 4. Cargar Retrieval V2

Se espera una lista de 50 casos con la estructura de retrieval utilizada previamente.

Para trazabilidad V2 también pueden incluirse:

- `filter_mode`
- `filters`
- `filter_trace`


In [6]:
if not RETRIEVAL_V2_INPUT_PATH.exists() or RETRIEVAL_V2_INPUT_PATH.stat().st_size == 0:
    retrieval_v2_batch = []
    print("⚠️ Sin corrida V2 todavía.")
else:
    with RETRIEVAL_V2_INPUT_PATH.open("r", encoding="utf-8-sig") as f:
        retrieval_v2_batch = json.load(f)

    assert isinstance(retrieval_v2_batch, list), (
        "retrieval_results_agentes_v2.json debe contener una lista de casos."
    )

    print("Casos recibidos:", len(retrieval_v2_batch))


Casos recibidos: 50


## 5. Validaciones duras de comparabilidad

El benchmark se detiene si:

- no llegan exactamente 50 casos;
- hay `case_id` duplicados;
- los casos no coinciden con el Ground Truth;
- alguna query difiere de la pregunta original;
- se detecta el fallback genérico.


In [7]:
if retrieval_v2_batch:
    gt_query_by_case = ground_truth_df.set_index("case_id")["pregunta"].to_dict()

    received_case_ids = [case.get("case_id") for case in retrieval_v2_batch]

    assert len(retrieval_v2_batch) == 50, (
        f"Se esperaban 50 casos y llegaron {len(retrieval_v2_batch)}."
    )
    assert len(set(received_case_ids)) == 50, "Hay case_id duplicados."
    assert set(received_case_ids) == set(gt_query_by_case), (
        "Los case_id recibidos no coinciden exactamente con el Ground Truth."
    )

    mismatches = []
    fallback_cases = []

    for case in retrieval_v2_batch:
        case_id = case["case_id"]
        query_sent = str(case.get("query", "")).strip()
        query_original = str(gt_query_by_case[case_id]).strip()

        if query_sent != query_original:
            mismatches.append(case_id)

        if is_fallback_query(query_sent):
            fallback_cases.append(case_id)

    assert not mismatches, (
        "Hay queries que no coinciden con el benchmark: " + ", ".join(mismatches)
    )
    assert not fallback_cases, (
        "Se detectó fallback genérico en: " + ", ".join(fallback_cases)
    )

    print("✅ 50/50 queries coinciden con el benchmark")
    print("✅ No se detectó fallback genérico")
else:
    print("⏳ Validación pendiente hasta recibir Retrieval V2.")


✅ 50/50 queries coinciden con el benchmark
✅ No se detectó fallback genérico


## 6. Construcción de resultados por caso


In [8]:
result_rows = []
trace_rows = []

gt_by_case = ground_truth_df.set_index("case_id").to_dict("index")

for case in retrieval_v2_batch:
    case_id = case["case_id"]
    gt = gt_by_case[case_id]

    status = case.get("status", "error")
    metric_eligible = status == "success"

    relevant_ids = parse_chunk_ids(gt["relevant_chunk_ids"])
    retrieved_ids = [
        item.get("chunk_id")
        for item in case.get("results", [])
        if item.get("chunk_id")
    ]

    query_original = str(gt["pregunta"]).strip()
    query_sent = str(case.get("query", "")).strip()

    filter_mode, filter_document_id, filter_categoria = infer_filter_trace(case)

    error = case.get("error") or {}
    error_code = error.get("code") if isinstance(error, dict) else None
    error_message = error.get("message") if isinstance(error, dict) else None

    if metric_eligible:
        r3 = recall_at_k(relevant_ids, retrieved_ids, 3)
        r5 = recall_at_k(relevant_ids, retrieved_ids, 5)
        p3 = precision_at_k(relevant_ids, retrieved_ids, 3)
        p5 = precision_at_k(relevant_ids, retrieved_ids, 5)
    else:
        r3 = r5 = p3 = p5 = None

    query_matches = query_original == query_sent
    fallback_used = is_fallback_query(query_sent)

    result_rows.append({
        "case_id": case_id,
        "document_id": gt["document_id"],
        "categoria": gt["categoria"],
        "pregunta": gt["pregunta"],
        "status": status,
        "metric_eligible": metric_eligible,
        "relevant_chunk_ids": relevant_ids,
        "retrieved_chunk_ids": retrieved_ids,
        "recall_at_3": r3,
        "recall_at_5": r5,
        "precision_at_3": p3,
        "precision_at_5": p5,
        "filter_mode": filter_mode,
        "filter_document_id": filter_document_id,
        "filter_categoria": filter_categoria,
        "query_original": query_original,
        "query_sent": query_sent,
        "query_matches_benchmark": query_matches,
        "fallback_used": fallback_used,
        "error_code": error_code,
        "error_message": error_message,
    })

    trace_rows.append({
        "case_id": case_id,
        "document_id": gt["document_id"],
        "categoria": gt["categoria"],
        "query_original": query_original,
        "query_sent": query_sent,
        "query_matches_benchmark": query_matches,
        "fallback_used": fallback_used,
        "filter_mode": filter_mode,
        "filter_document_id": filter_document_id,
        "filter_categoria": filter_categoria,
        "status": status,
    })

results_v2_df = pd.DataFrame(result_rows)
query_trace_df = pd.DataFrame(trace_rows)

results_v2_df.head() if not results_v2_df.empty else results_v2_df


,case_id,document_id,categoria,pregunta,status,metric_eligible,relevant_chunk_ids,retrieved_chunk_ids,recall_at_3,recall_at_5,...,precision_at_5,filter_mode,filter_document_id,filter_categoria,query_original,query_sent,query_matches_benchmark,fallback_used,error_code,error_message
0,AI-ES-001-Q01,AI-ES-001,IA,¿Qué pretende introducir este módulo sobre int...,success,True,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-001_CH_002]",1.0,1.0,...,0.2,document_id,AI-ES-001,None,¿Qué pretende introducir este módulo sobre int...,¿Qué pretende introducir este módulo sobre int...,True,False,None,None
1,AI-ES-001-Q02,AI-ES-001,IA,¿Es obligatorio conocer aprendizaje automático...,success,True,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-001_CH_002]",1.0,1.0,...,0.2,document_id,AI-ES-001,None,¿Es obligatorio conocer aprendizaje automático...,¿Es obligatorio conocer aprendizaje automático...,True,False,None,None
2,AI-ES-001-Q03,AI-ES-001,IA,Menciona dos cargas de trabajo de IA tratadas ...,success,True,[AI-ES-001_CH_002],"[AI-ES-001_CH_002, AI-ES-001_CH_001]",1.0,1.0,...,0.2,document_id,AI-ES-001,None,Menciona dos cargas de trabajo de IA tratadas ...,Menciona dos cargas de trabajo de IA tratadas ...,True,False,None,None
3,AI-ES-001-Q04,AI-ES-001,IA,¿Qué tema del módulo aborda el uso ético o seg...,success,True,[AI-ES-001_CH_002],"[AI-ES-001_CH_002, AI-ES-001_CH_001]",1.0,1.0,...,0.2,document_id,AI-ES-001,None,¿Qué tema del módulo aborda el uso ético o seg...,¿Qué tema del módulo aborda el uso ético o seg...,True,False,None,None
4,AI-ES-001-Q05,AI-ES-001,IA,¿Qué conocimiento previo puede ser útil antes ...,success,True,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-001_CH_002]",1.0,1.0,...,0.2,document_id,AI-ES-001,None,¿Qué conocimiento previo puede ser útil antes ...,¿Qué conocimiento previo puede ser útil antes ...,True,False,None,None


## 7. Métricas globales y por categoría


In [9]:
metric_cols = [
    "recall_at_3",
    "recall_at_5",
    "precision_at_3",
    "precision_at_5",
]

if not results_v2_df.empty:
    eligible_df = results_v2_df[results_v2_df["metric_eligible"] == True].copy()

    metrics_summary_df = pd.DataFrame([{
        "benchmark_run": manifest["benchmark_run"],
        "total_cases": len(results_v2_df),
        "successful_cases": int((results_v2_df["status"] == "success").sum()),
        "no_results_cases": int((results_v2_df["status"] == "no_results").sum()),
        "error_cases": int((results_v2_df["status"] == "error").sum()),
        **{col: eligible_df[col].mean() for col in metric_cols},
    }])

    category_v2 = (
        eligible_df
        .groupby("categoria", as_index=False)
        .agg(
            total_cases=("case_id", "count"),
            successful_cases=("case_id", "count"),
            recall_at_3=("recall_at_3", "mean"),
            recall_at_5=("recall_at_5", "mean"),
            precision_at_3=("precision_at_3", "mean"),
            precision_at_5=("precision_at_5", "mean"),
        )
    )

    display(metrics_summary_df)
    display(category_v2)
else:
    metrics_summary_df = pd.DataFrame()
    category_v2 = pd.DataFrame()
    print("⏳ Métricas pendientes hasta recibir Retrieval V2.")


,benchmark_run,total_cases,successful_cases,no_results_cases,error_cases,recall_at_3,recall_at_5,precision_at_3,precision_at_5
0,retrieval_v2_gt_v2,50,50,0,0,0.933333,0.953333,0.34,0.208


,categoria,total_cases,successful_cases,recall_at_3,recall_at_5,precision_at_3,precision_at_5
0,Backend,10,10,0.866667,0.866667,0.366667,0.22
1,Cloud / DevOps,10,10,0.900000,0.900000,0.300000,0.18
2,Data Science,10,10,1.000000,1.000000,0.333333,0.20
3,Frontend,10,10,0.900000,1.000000,0.333333,0.22
4,IA,10,10,1.000000,1.000000,0.366667,0.22


## 8. Comparación V1 vs V2 por categoría


In [10]:
baseline_category_df = pd.read_csv(
    BASELINE_CATEGORY_PATH,
    encoding="utf-8-sig"
)

if not category_v2.empty:
    category_comparison_df = category_v2.merge(
        baseline_category_df,
        on="categoria",
        how="left",
        suffixes=("", "_v1")
    )

    for metric in metric_cols:
        category_comparison_df[f"delta_{metric}"] = (
            category_comparison_df[metric]
            - category_comparison_df[f"{metric}_v1"]
        )

    delta_cols = [f"delta_{m}" for m in metric_cols]
    category_comparison_df["regression_detected"] = (
        category_comparison_df[delta_cols].lt(0).any(axis=1)
    )

    ordered_cols = [
        "categoria",
        "total_cases",
        "successful_cases",
        "recall_at_3",
        "recall_at_5",
        "precision_at_3",
        "precision_at_5",
        "recall_at_3_v1",
        "recall_at_5_v1",
        "precision_at_3_v1",
        "precision_at_5_v1",
        "delta_recall_at_3",
        "delta_recall_at_5",
        "delta_precision_at_3",
        "delta_precision_at_5",
        "regression_detected",
    ]

    category_comparison_df = category_comparison_df[ordered_cols]
    display(category_comparison_df)
else:
    category_comparison_df = pd.DataFrame()


,categoria,total_cases,successful_cases,recall_at_3,recall_at_5,precision_at_3,precision_at_5,recall_at_3_v1,recall_at_5_v1,precision_at_3_v1,precision_at_5_v1,delta_recall_at_3,delta_recall_at_5,delta_precision_at_3,delta_precision_at_5,regression_detected
0,Backend,10,10,0.866667,0.866667,0.366667,0.22,0.516667,0.716667,0.233333,0.18,0.35,0.15,0.133333,0.04,False
1,Cloud / DevOps,10,10,0.900000,0.900000,0.300000,0.18,0.800000,0.800000,0.266667,0.16,0.10,0.10,0.033333,0.02,False
2,Data Science,10,10,1.000000,1.000000,0.333333,0.20,0.600000,0.800000,0.200000,0.16,0.40,0.20,0.133333,0.04,False
3,Frontend,10,10,0.900000,1.000000,0.333333,0.22,0.350000,0.550000,0.133333,0.12,0.55,0.45,0.200000,0.10,False
4,IA,10,10,1.000000,1.000000,0.366667,0.22,0.600000,0.900000,0.233333,0.20,0.40,0.10,0.133333,0.02,False


In [11]:
regression_rows = []

if not category_comparison_df.empty:
    for _, row in category_comparison_df.iterrows():
        for metric in metric_cols:
            v1 = row[f"{metric}_v1"]
            v2 = row[metric]
            delta = v2 - v1

            regression_rows.append({
                "scope": "categoria",
                "name": row["categoria"],
                "metric": metric,
                "v1_value": v1,
                "v2_value": v2,
                "delta": delta,
                "regression": bool(delta < 0),
                "severity": (
                    "high" if delta <= -0.10
                    else "medium" if delta <= -0.05
                    else "low" if delta < 0
                    else "none"
                ),
                "notes": "",
            })

regression_df = pd.DataFrame(regression_rows)
regression_df[regression_df["regression"] == True] if not regression_df.empty else regression_df


,scope,name,metric,v1_value,v2_value,delta,regression,severity,notes


## 9. Seguimiento de los 11 casos críticos


In [12]:
baseline_results_df = pd.read_csv(
    BASELINE_RESULTS_PATH,
    encoding="utf-8-sig"
)

baseline_errors_df = pd.read_csv(
    BASELINE_ERROR_ANALYSIS_PATH,
    encoding="utf-8-sig"
)

assert len(baseline_errors_df) == manifest["critical_cases_expected"], (
    f"Se esperaban {manifest['critical_cases_expected']} casos críticos "
    f"y se encontraron {len(baseline_errors_df)}."
)

print("Casos críticos V1:", len(baseline_errors_df))
baseline_errors_df[
    ["case_id", "categoria", "causa_raiz", "recall_at_5"]
]


Casos críticos V1: 11


,case_id,categoria,causa_raiz,recall_at_5
0,BE-ES-001-Q02,Backend,RANKING_SEMANTICO,0.0
1,BE-ES-002-Q04,Backend,ENTIDAD_LEXICAL,0.0
2,CLD-ES-002-Q02,Cloud / DevOps,AMBIGUEDAD_CONTEXTO,0.0
3,CLD-ES-002-Q03,Cloud / DevOps,COMPETENCIA_INTRADOCUMENTO,0.0
4,DS-ES-001-Q04,Data Science,AMBIGUEDAD_CONTEXTO,0.0
5,DS-ES-002-Q05,Data Science,AMBIGUEDAD_CONTEXTO,0.0
6,FE-ES-001-Q03,Frontend,COMPETENCIA_INTRADOCUMENTO,0.0
7,FE-ES-001-Q05,Frontend,RANKING_SEMANTICO,0.0
8,FE-ES-002-Q03,Frontend,RANKING_SEMANTICO,0.0
9,FE-ES-002-Q05,Frontend,PREGUNTA_INFERENCIAL,0.0


In [13]:
critical_rows = []

if not results_v2_df.empty:
    baseline_by_case = baseline_results_df.set_index("case_id").to_dict("index")
    v2_by_case = results_v2_df.set_index("case_id").to_dict("index")

    for _, critical in baseline_errors_df.iterrows():
        case_id = critical["case_id"]
        v1 = baseline_by_case[case_id]
        v2 = v2_by_case[case_id]

        recall5_v1 = float(v1["recall_at_5"])
        recall5_v2 = float(v2["recall_at_5"]) if pd.notna(v2["recall_at_5"]) else None
        precision5_v1 = float(v1["precision_at_5"])
        precision5_v2 = float(v2["precision_at_5"]) if pd.notna(v2["precision_at_5"]) else None

        resolved = (
            recall5_v1 == 0
            and recall5_v2 is not None
            and recall5_v2 > 0
        )

        critical_rows.append({
            "case_id": case_id,
            "document_id": v2["document_id"],
            "categoria": v2["categoria"],
            "pregunta": v2["pregunta"],
            "root_cause_v1": critical["causa_raiz"],
            "relevant_chunk_ids": v2["relevant_chunk_ids"],
            "retrieved_chunk_ids_v1": parse_chunk_ids(v1["retrieved_chunk_ids"]),
            "retrieved_chunk_ids_v2": v2["retrieved_chunk_ids"],
            "recall_at_3_v1": v1["recall_at_3"],
            "recall_at_3_v2": v2["recall_at_3"],
            "recall_at_5_v1": recall5_v1,
            "recall_at_5_v2": recall5_v2,
            "precision_at_3_v1": v1["precision_at_3"],
            "precision_at_3_v2": v2["precision_at_3"],
            "precision_at_5_v1": precision5_v1,
            "precision_at_5_v2": precision5_v2,
            "filter_mode": v2["filter_mode"],
            "expected_improvement": critical["recomendacion"],
            "observed_improvement": (
                "Sí" if resolved
                else "Parcial" if recall5_v2 is not None and recall5_v2 > recall5_v1
                else "No"
            ),
            "resolved": resolved,
            "notes": "",
        })

critical_cases_df = pd.DataFrame(critical_rows)

if not critical_cases_df.empty:
    print(
        "Casos críticos que dejaron Recall@5 = 0:",
        int(critical_cases_df["resolved"].sum()),
        "de",
        len(critical_cases_df),
    )
    display(critical_cases_df)
else:
    critical_cases_df = pd.DataFrame()


Casos críticos que dejaron Recall@5 = 0: 11 de 11


,case_id,document_id,categoria,pregunta,root_cause_v1,relevant_chunk_ids,retrieved_chunk_ids_v1,retrieved_chunk_ids_v2,recall_at_3_v1,recall_at_3_v2,...,recall_at_5_v2,precision_at_3_v1,precision_at_3_v2,precision_at_5_v1,precision_at_5_v2,filter_mode,expected_improvement,observed_improvement,resolved,notes
0,BE-ES-001-Q02,BE-ES-001,Backend,¿Qué representa la variable app en el ejemplo ...,RANKING_SEMANTICO,[BE-ES-001_CH_011],"[BE-ES-002_CH_014, FE-ES-002_CH_005, BE-ES-001...","[BE-ES-001_CH_011, BE-ES-001_CH_007, BE-ES-001...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Probar retrieval híbrido y mayor peso a tokens...,Sí,True,
1,BE-ES-002-Q04,BE-ES-002,Backend,¿Qué archivo contiene los ajustes o configurac...,ENTIDAD_LEXICAL,[BE-ES-002_CH_008],"[BE-ES-002_CH_004, BE-ES-002_CH_002, BE-ES-002...","[BE-ES-002_CH_008, BE-ES-002_CH_004, BE-ES-002...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Combinar similitud vectorial con búsqueda lexi...,Sí,True,
2,CLD-ES-002-Q02,CLD-ES-002,Cloud / DevOps,¿Qué conocimientos previos se piden?,AMBIGUEDAD_CONTEXTO,[CLD-ES-002_CH_003],"[FE-ES-002_CH_002, AI-ES-002_CH_002, DS-ES-002...","[CLD-ES-002_CH_003, CLD-ES-002_CH_004, CLD-ES-...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Añadir contexto de la ruta Cloud o prefiltrar ...,Sí,True,
3,CLD-ES-002-Q03,CLD-ES-002,Cloud / DevOps,¿Qué temas cubre el módulo 'Descripción de la ...,COMPETENCIA_INTRADOCUMENTO,[CLD-ES-002_CH_003],"[CLD-ES-002_CH_004, CLD-ES-002_CH_001, CLD-ES-...","[CLD-ES-002_CH_001, CLD-ES-002_CH_004, CLD-ES-...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Reranking con títulos/secciones y revisar los ...,Sí,True,
4,DS-ES-001-Q04,DS-ES-001,Data Science,¿Qué ejercicio práctico aparece en el módulo?,AMBIGUEDAD_CONTEXTO,[DS-ES-001_CH_001],"[DS-ES-002_CH_002, DS-ES-001_CH_002, DS-ES-002...","[DS-ES-001_CH_001, DS-ES-001_CH_002]",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Agregar contexto documental o prefiltrar por d...,Sí,True,
5,DS-ES-002-Q05,DS-ES-002,Data Science,¿Qué requisito previo se indica?,AMBIGUEDAD_CONTEXTO,[DS-ES-002_CH_001],"[FE-ES-002_CH_002, DS-ES-001_CH_002, CLD-ES-00...","[DS-ES-002_CH_001, DS-ES-002_CH_002]",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Prefiltrar por documento/categoría o contextua...,Sí,True,
6,FE-ES-001-Q03,FE-ES-001,Frontend,¿Qué aprende el lector sobre los elementos HTML?,COMPETENCIA_INTRADOCUMENTO,[FE-ES-001_CH_001],"[FE-ES-001_CH_028, FE-ES-001_CH_049, FE-ES-001...","[FE-ES-001_CH_049, FE-ES-001_CH_029, FE-ES-001...",0.0,0.0,...,1.0,0.0,0.000000,0.0,0.2,document_id,Aplicar reranking y conservar título/sección d...,Sí,True,
7,FE-ES-001-Q05,FE-ES-001,Frontend,¿Qué puede hacer una etiqueta HTML con el cont...,RANKING_SEMANTICO,[FE-ES-001_CH_002],"[FE-ES-001_CH_028, FE-ES-001_CH_013, FE-ES-001...","[FE-ES-001_CH_002, FE-ES-001_CH_005, FE-ES-001...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Probar híbrido BM25 + vectorial y reranking.,Sí,True,
8,FE-ES-002-Q03,FE-ES-002,Frontend,¿Para qué sirven los bucles en JavaScript?,RANKING_SEMANTICO,[FE-ES-002_CH_003],"[FE-ES-002_CH_007, FE-ES-001_CH_047, FE-ES-001...","[FE-ES-002_CH_007, FE-ES-002_CH_001, FE-ES-002...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Favorecer coincidencias lexicales 'bucles' + '...,Sí,True,
9,FE-ES-002-Q05,FE-ES-002,Frontend,¿El módulo se limita únicamente a sintaxis bás...,PREGUNTA_INFERENCIAL,[FE-ES-002_CH_001],"[DS-ES-001_CH_002, DS-ES-002_CH_002, FE-ES-002...","[FE-ES-002_CH_004, FE-ES-002_CH_002, FE-ES-002...",0.0,1.0,...,1.0,0.0,0.333333,0.0,0.2,document_id,Revisar query de evaluación y probar reranking...,Sí,True,


## 10. Impacto de filtros en los casos críticos


In [14]:
filter_rows = []

if not critical_cases_df.empty:
    for _, row in critical_cases_df.iterrows():
        delta_r5 = row["recall_at_5_v2"] - row["recall_at_5_v1"]
        delta_p5 = row["precision_at_5_v2"] - row["precision_at_5_v1"]

        filter_rows.append({
            "case_id": row["case_id"],
            "document_id": row["document_id"],
            "categoria": row["categoria"],
            "root_cause_v1": row["root_cause_v1"],
            "filter_mode": row["filter_mode"],
            "recall_at_5_v1": row["recall_at_5_v1"],
            "recall_at_5_v2": row["recall_at_5_v2"],
            "precision_at_5_v1": row["precision_at_5_v1"],
            "precision_at_5_v2": row["precision_at_5_v2"],
            "delta_recall_at_5": delta_r5,
            "delta_precision_at_5": delta_p5,
            "improved": bool(delta_r5 > 0 or delta_p5 > 0),
        })

filter_impact_df = pd.DataFrame(filter_rows)

if not filter_impact_df.empty:
    display(
        filter_impact_df[
            filter_impact_df["root_cause_v1"].isin([
                "AMBIGUEDAD_CONTEXTO",
                "COMPETENCIA_INTRADOCUMENTO",
            ])
        ]
    )
else:
    filter_impact_df = pd.DataFrame()


,case_id,document_id,categoria,root_cause_v1,filter_mode,recall_at_5_v1,recall_at_5_v2,precision_at_5_v1,precision_at_5_v2,delta_recall_at_5,delta_precision_at_5,improved
2,CLD-ES-002-Q02,CLD-ES-002,Cloud / DevOps,AMBIGUEDAD_CONTEXTO,document_id,0.0,1.0,0.0,0.2,1.0,0.2,True
3,CLD-ES-002-Q03,CLD-ES-002,Cloud / DevOps,COMPETENCIA_INTRADOCUMENTO,document_id,0.0,1.0,0.0,0.2,1.0,0.2,True
4,DS-ES-001-Q04,DS-ES-001,Data Science,AMBIGUEDAD_CONTEXTO,document_id,0.0,1.0,0.0,0.2,1.0,0.2,True
5,DS-ES-002-Q05,DS-ES-002,Data Science,AMBIGUEDAD_CONTEXTO,document_id,0.0,1.0,0.0,0.2,1.0,0.2,True
6,FE-ES-001-Q03,FE-ES-001,Frontend,COMPETENCIA_INTRADOCUMENTO,document_id,0.0,1.0,0.0,0.2,1.0,0.2,True
10,AI-ES-001-Q05,AI-ES-001,IA,AMBIGUEDAD_CONTEXTO,document_id,0.0,1.0,0.0,0.2,1.0,0.2,True


## 11. Análisis de fallos que permanecen


In [15]:
error_rows = []

if not results_v2_df.empty:
    error_map = baseline_errors_df.set_index("case_id").to_dict("index")

    failing = results_v2_df[
        (results_v2_df["recall_at_5"].fillna(0) == 0)
        | (results_v2_df["metric_eligible"] == False)
    ]

    for _, row in failing.iterrows():
        previous = error_map.get(row["case_id"], {})

        error_rows.append({
            "case_id": row["case_id"],
            "document_id": row["document_id"],
            "categoria": row["categoria"],
            "pregunta": row["pregunta"],
            "causa_raiz_v1": previous.get("causa_raiz", ""),
            "causa_raiz_v2": "",
            "evidencia": "",
            "recomendacion": "",
            "relevant_chunk_ids": row["relevant_chunk_ids"],
            "retrieved_chunk_ids": row["retrieved_chunk_ids"],
            "recall_at_3": row["recall_at_3"],
            "recall_at_5": row["recall_at_5"],
            "precision_at_3": row["precision_at_3"],
            "precision_at_5": row["precision_at_5"],
            "filter_mode": row["filter_mode"],
            "resolved_vs_v1": False,
        })

error_analysis_df = pd.DataFrame(error_rows)
error_analysis_df


,case_id,document_id,categoria,pregunta,causa_raiz_v1,causa_raiz_v2,evidencia,recomendacion,relevant_chunk_ids,retrieved_chunk_ids,recall_at_3,recall_at_5,precision_at_3,precision_at_5,filter_mode,resolved_vs_v1
0,BE-ES-002-Q03,BE-ES-002,Backend,¿Qué comando crea el proyecto mysite dentro de...,,,,,[BE-ES-002_CH_005],"[BE-ES-002_CH_006, BE-ES-002_CH_010, BE-ES-002...",0.0,0.0,0.0,0.0,document_id,False
1,CLD-ES-001-Q05,CLD-ES-001,Cloud / DevOps,Menciona dos formas de describir a Kubernetes ...,,,,,[CLD-ES-001_CH_002],"[CLD-ES-001_CH_003, CLD-ES-001_CH_014, CLD-ES-...",0.0,0.0,0.0,0.0,document_id,False


## 12. Exportar artefactos Retrieval V2


In [16]:
OUTPUT_FILES = {
    "metrics": OUTPUT_DIR / "retrieval_metrics_gt_v2.csv",
    "results": OUTPUT_DIR / "retrieval_results_gt_v2.csv",
    "query_trace": OUTPUT_DIR / "retrieval_query_trace_gt_v2.csv",
    "category": OUTPUT_DIR / "retrieval_category_summary_gt_v2.csv",
    "critical": OUTPUT_DIR / "retrieval_critical_cases_gt_v2.csv",
    "regression": OUTPUT_DIR / "retrieval_regression_analysis_gt_v2.csv",
    "errors": OUTPUT_DIR / "retrieval_error_analysis_gt_v2.csv",
    "filters": OUTPUT_DIR / "retrieval_filter_impact_gt_v2.csv",
}

if results_v2_df.empty:
    print("⏳ No se exportan resultados porque Retrieval V2 todavía no fue cargado.")
else:
    assert len(results_v2_df) == 50
    assert results_v2_df["query_matches_benchmark"].all()
    assert not results_v2_df["fallback_used"].any()

    metrics_summary_df.to_csv(
        OUTPUT_FILES["metrics"], index=False, encoding="utf-8-sig"
    )
    results_v2_df.to_csv(
        OUTPUT_FILES["results"], index=False, encoding="utf-8-sig"
    )
    query_trace_df.to_csv(
        OUTPUT_FILES["query_trace"], index=False, encoding="utf-8-sig"
    )
    category_comparison_df.to_csv(
        OUTPUT_FILES["category"], index=False, encoding="utf-8-sig"
    )
    critical_cases_df.to_csv(
        OUTPUT_FILES["critical"], index=False, encoding="utf-8-sig"
    )
    regression_df.to_csv(
        OUTPUT_FILES["regression"], index=False, encoding="utf-8-sig"
    )
    error_analysis_df.to_csv(
        OUTPUT_FILES["errors"], index=False, encoding="utf-8-sig"
    )
    filter_impact_df.to_csv(
        OUTPUT_FILES["filters"], index=False, encoding="utf-8-sig"
    )

    print("✅ Artefactos exportados:")
    for name, path in OUTPUT_FILES.items():
        print(f"  - {name}: {path}")


✅ Artefactos exportados:
  - metrics: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_metrics_gt_v2.csv
  - results: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_results_gt_v2.csv
  - query_trace: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_query_trace_gt_v2.csv
  - category: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_category_summary_gt_v2.csv
  - critical: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_critical_cases_gt_v2.csv
  - regression: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_regression_analysis_gt_v2.csv
  - errors: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retrieval_error_analysis_gt_v2.csv
  - filters: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2\retr

## 13. Resumen automático V1 vs V2


In [17]:
if not results_v2_df.empty:
    baseline_global = manifest["baseline_reference"]
    current = metrics_summary_df.iloc[0]

    summary = pd.DataFrame([
        {
            "metric": "Recall@3",
            "V1": baseline_global["recall_at_3"],
            "V2": current["recall_at_3"],
            "delta": current["recall_at_3"] - baseline_global["recall_at_3"],
        },
        {
            "metric": "Recall@5",
            "V1": baseline_global["recall_at_5"],
            "V2": current["recall_at_5"],
            "delta": current["recall_at_5"] - baseline_global["recall_at_5"],
        },
        {
            "metric": "Precision@3",
            "V1": baseline_global["precision_at_3"],
            "V2": current["precision_at_3"],
            "delta": current["precision_at_3"] - baseline_global["precision_at_3"],
        },
        {
            "metric": "Precision@5",
            "V1": baseline_global["precision_at_5"],
            "V2": current["precision_at_5"],
            "delta": current["precision_at_5"] - baseline_global["precision_at_5"],
        },
    ])

    display(summary)

    print(
        "\nCasos críticos resueltos:",
        int(critical_cases_df["resolved"].sum()),
        "/",
        len(critical_cases_df),
    )

    regressions = (
        regression_df[regression_df["regression"] == True]
        if not regression_df.empty
        else pd.DataFrame()
    )
    print("Regresiones por categoría/métrica:", len(regressions))
else:
    print("⏳ Resumen final pendiente hasta ejecutar Retrieval V2.")


,metric,V1,V2,delta
0,Recall@3,0.573333,0.933333,0.360000
1,Recall@5,0.753333,0.953333,0.200000
2,Precision@3,0.213333,0.340000,0.126667
3,Precision@5,0.164000,0.208000,0.044000



Casos críticos resueltos: 11 / 11
Regresiones por categoría/métrica: 0


## Checklist de cierre

- [ ] Se recibieron exactamente 50 casos.
- [ ] Las 50 queries coinciden con `ground_truth_v2.pregunta`.
- [ ] No se utilizó fallback genérico.
- [ ] Se documentó el modo de filtro por caso.
- [ ] Se calcularon Recall@3, Recall@5, Precision@3 y Precision@5.
- [ ] Se revisaron los 11 casos críticos.
- [ ] Se verificaron regresiones por categoría.
- [ ] Se documentó el `score_type` correcto de Retrieval V2.
- [ ] Los artefactos V1 permanecen intactos.
